## ✈️ Use Case 1: Flight Delay Prediction
**Goal:** Predict flight arrival delay (minutes) and/or whether a flight is delayed (>15 min) using PySpark MLlib.  
_

### Step 1: Load & inspect the data

In [0]:
df = spark.table("workspace.default.`1_airlines_delay`")
display(df.limit(5))
df.printSchema()

Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,UniqueCarrier,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ArrDelay,DepDelay,Origin,Dest,Distance,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
2008,1,3,4,2003.0,1955,2211.0,2225,WN,335,N712SW,128.0,150.0,116.0,-14.0,8.0,IAD,TPA,810,4.0,8.0,0,N,0,null,null,null,null,null
2008,1,3,4,754.0,735,1002.0,1000,WN,3231,N772SW,128.0,145.0,113.0,2.0,19.0,IAD,TPA,810,5.0,10.0,0,N,0,null,null,null,null,null
2008,1,3,4,628.0,620,804.0,750,WN,448,N428WN,96.0,90.0,76.0,14.0,8.0,IND,BWI,515,3.0,17.0,0,N,0,null,null,null,null,null
2008,1,3,4,1829.0,1755,1959.0,1925,WN,3920,N464WN,90.0,90.0,77.0,34.0,34.0,IND,BWI,515,3.0,10.0,0,N,0,2.0,0.0,0.0,0.0,32.0
2008,1,3,4,1940.0,1915,2121.0,2110,WN,378,N726SW,101.0,115.0,87.0,11.0,25.0,IND,JAX,688,4.0,10.0,0,N,0,null,null,null,null,null


root
 |-- Year: long (nullable = true)
 |-- Month: long (nullable = true)
 |-- DayofMonth: long (nullable = true)
 |-- DayOfWeek: long (nullable = true)
 |-- DepTime: double (nullable = true)
 |-- CRSDepTime: long (nullable = true)
 |-- ArrTime: double (nullable = true)
 |-- CRSArrTime: long (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- FlightNum: long (nullable = true)
 |-- TailNum: string (nullable = true)
 |-- ActualElapsedTime: double (nullable = true)
 |-- CRSElapsedTime: double (nullable = true)
 |-- AirTime: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- Distance: long (nullable = true)
 |-- TaxiIn: double (nullable = true)
 |-- TaxiOut: double (nullable = true)
 |-- Cancelled: long (nullable = true)
 |-- CancellationCode: string (nullable = true)
 |-- Diverted: long (nullable = true)
 |-- CarrierDelay: double (nullable = 

### Step 2: Rename key columns for _consistency_

In [0]:
# Rename and standardize column names
df = (df
    .withColumnRenamed("UniqueCarrier", "AIRLINE")
    .withColumnRenamed("ArrDelay", "ARRIVAL_DELAY")
    .withColumnRenamed("DepDelay", "DEPARTURE_DELAY")
    .withColumnRenamed("Origin", "ORIGIN")
    .withColumnRenamed("Dest", "DEST")
    .withColumnRenamed("Distance", "DISTANCE")
)

print(df.columns)


['Year', 'Month', 'DayofMonth', 'DayOfWeek', 'DepTime', 'CRSDepTime', 'ArrTime', 'CRSArrTime', 'AIRLINE', 'FlightNum', 'TailNum', 'ActualElapsedTime', 'CRSElapsedTime', 'AirTime', 'ARRIVAL_DELAY', 'DEPARTURE_DELAY', 'ORIGIN', 'DEST', 'DISTANCE', 'TaxiIn', 'TaxiOut', 'Cancelled', 'CancellationCode', 'Diverted', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']


Step 3: Clean and filter data

In [0]:
from pyspark.sql.functions import col, when
from pyspark.sql.types import DoubleType

df = (df
      .withColumn("ARRIVAL_DELAY", col("ARRIVAL_DELAY").cast(DoubleType()))
      .withColumn("DEPARTURE_DELAY", col("DEPARTURE_DELAY").cast(DoubleType()))
      .withColumn("DISTANCE", col("DISTANCE").cast(DoubleType()))
      .filter((col("Cancelled") == 0) & (col("Diverted") == 0))
      .filter((col("ARRIVAL_DELAY") > -100) & (col("ARRIVAL_DELAY") < 1000))
)

display(df.limit(5))


Year,Month,DayofMonth,DayOfWeek,DepTime,CRSDepTime,ArrTime,CRSArrTime,AIRLINE,FlightNum,TailNum,ActualElapsedTime,CRSElapsedTime,AirTime,ARRIVAL_DELAY,DEPARTURE_DELAY,ORIGIN,DEST,DISTANCE,TaxiIn,TaxiOut,Cancelled,CancellationCode,Diverted,CarrierDelay,WeatherDelay,NASDelay,SecurityDelay,LateAircraftDelay
2008,6,7,6,1800.0,1740,2114.0,2025,AA,1659,N539AA,194.0,165.0,178.0,49.0,20.0,SAT,ORD,1041.0,7.0,9.0,0,N,0,0.0,0.0,35.0,0.0,14.0
2008,6,8,7,1903.0,1740,2135.0,2025,AA,1659,N579AA,152.0,165.0,130.0,70.0,83.0,SAT,ORD,1041.0,12.0,10.0,0,N,0,0.0,0.0,25.0,0.0,45.0
2008,6,9,1,1819.0,1740,2113.0,2025,AA,1659,N499AA,174.0,165.0,144.0,48.0,39.0,SAT,ORD,1041.0,7.0,23.0,0,N,0,0.0,0.0,18.0,0.0,30.0
2008,6,11,3,1926.0,1740,2203.0,2025,AA,1659,N584AA,157.0,165.0,130.0,98.0,106.0,SAT,ORD,1041.0,20.0,7.0,0,N,0,0.0,0.0,0.0,0.0,98.0
2008,6,12,4,1917.0,1740,2156.0,2025,AA,1659,N292AA,159.0,165.0,136.0,91.0,97.0,SAT,ORD,1041.0,8.0,15.0,0,N,0,0.0,0.0,57.0,0.0,34.0


### Step 4: Feature engineering

In [0]:
df = df.withColumn("DELAYED", when(col("ARRIVAL_DELAY") > 15, 1).otherwise(0))
display(df.select("AIRLINE", "ARRIVAL_DELAY", "DELAYED").limit(5))


AIRLINE,ARRIVAL_DELAY,DELAYED
AA,49.0,1
AA,70.0,1
AA,48.0,1
AA,98.0,1
AA,91.0,1


### Step 5: SQL exploratory analysis

In [0]:
df.createOrReplaceTempView("flights")

# Average delay per airline
spark.sql("""
  SELECT AIRLINE, COUNT(*) AS total_flights,
         ROUND(AVG(ARRIVAL_DELAY),2) AS avg_arr_delay,
         ROUND(SUM(DELAYED)/COUNT(*)*100,2) AS pct_delayed
  FROM flights
  GROUP BY AIRLINE
  ORDER BY avg_arr_delay DESC
""").show(10)


+-------+-------------+-------------+-----------+
|AIRLINE|total_flights|avg_arr_delay|pct_delayed|
+-------+-------------+-------------+-----------+
|     YV|        66769|        55.29|      74.19|
|     B6|        54925|        55.09|      68.34|
|     OH|        52453|        51.02|      73.14|
|     XE|       103147|        50.18|      68.31|
|     UA|       140898|        47.74|      66.25|
|     EV|        81762|        47.55|      67.86|
|     9E|        51565|        46.86|      67.55|
|     AA|       190844|        46.19|      67.77|
|     OO|       131780|        45.37|      65.73|
|     MQ|       141192|        45.05|      67.35|
+-------+-------------+-------------+-----------+
only showing top 10 rows


### Step 6: Prepare ML Features

In [0]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

indexer = StringIndexer(inputCol="AIRLINE", outputCol="AIRLINE_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCols=["AIRLINE_idx"], outputCols=["AIRLINE_ohe"])
assembler = VectorAssembler(
    inputCols=["DEPARTURE_DELAY", "DISTANCE", "DayOfWeek", "AIRLINE_ohe"],
    outputCol="features"
)
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")


### Step 7: Model training (Logistic Regression)
### # 
We’ll predict DELAYED using Logistic Regression (binary classification).

In [0]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="scaledFeatures", labelCol="DELAYED")
pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, lr])

train, test = df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)
pred = model.transform(test)


### Step 8: Model evaluation

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="DELAYED", metricName="areaUnderROC")
auc = evaluator.evaluate(pred)
print("ROC AUC:", round(auc, 3))

pred.createOrReplaceTempView("predictions")
spark.sql("""
SELECT prediction, DELAYED as actual, COUNT(*) as count
FROM predictions
GROUP BY prediction, DELAYED
ORDER BY count DESC
""").show()


ROC AUC: 0.915
+----------+------+------+
|prediction|actual| count|
+----------+------+------+
|       1.0|     1|204477|
|       0.0|     0|119040|
|       0.0|     1| 38045|
|       1.0|     0| 23681|
+----------+------+------+



### Step 9: Save model and results

In [0]:
pred.select("Year","Month","AIRLINE","ORIGIN","DEST","DELAYED","prediction") \
    .write.mode("overwrite").saveAsTable("flight_delay_predictions")

print("✅ Predictions saved as SQL table: flight_delay_predictions")


✅ Predictions saved as SQL table: flight_delay_predictions


### Step 10: Flight Delay Prediction results

In [0]:
%sql
SELECT AIRLINE, ROUND(AVG(prediction),2) AS delay_probability
FROM flight_delay_predictions
GROUP BY AIRLINE
ORDER BY delay_probability DESC;


AIRLINE,delay_probability
YV,0.72
OH,0.7
EV,0.66
AA,0.65
B6,0.65
9E,0.65
MQ,0.65
XE,0.65
NW,0.65
UA,0.63


Databricks visualization. Run in Databricks to view.